In [ ]:
import os 
import pandas as pd 
import ast
from mcherry_functions import load_mch_image,mch_int
from caiman.utils.visualization import get_contours
import sys
sys.path.extend(['/Users/amonast/Documents/GitHub/Engram_2P/Engram_2P',
                 '/projectnb/sramirezlab/amonast/Engram_2P/Engram_2P'])
from spatial.rois import get_h5,load_cnmf
import numpy as np
import panel as pn
import tifffile

## get data & mCherry threshold

In [ ]:

base_dir = '/projectnb/sramirezlab/amonast/Tone2P'
file_key = '/projectnb/sramirezlab/amonast/Tone2P/Data_info_TFC.csv'
ani = 'F5L'
fov='FOV1' 
info = pd.read_csv(file_key)
sessions = info['Session'].loc[info['Animal']==ani].unique()
ind_csv=os.path.join(base_dir,'Tagging','CellReg_output',ani+'_'+fov+'_reg_indices.csv')
#ind_df = pd.read_csv(ind_csv).drop(['Unnamed: 0'],axis=1).astype(int)

thr_df = pd.read_csv(os.path.join(base_dir,'Tagging','mcherry_thresholds.csv'))
thr = thr_df['mch_thr'].loc[(thr_df['Animal']==ani) & (thr_df['FOV']==fov)].values[0]
print('threshold:'+str(thr))

fiji_roi_savepath = os.path.join(base_dir,'ROIs','all_rois',ani+'_'+fov)
os.makedirs(fiji_roi_savepath,exist_ok=True)

In [ ]:
df = pd.DataFrame(pd.read_csv(ind_csv,index_col=0))
# Convert the string representations of lists into actual lists using ast.literal_eval
df['mch_pre'] = df['mch_pre'].apply(ast.literal_eval)
df['mch_post'] = df['mch_post'].apply(ast.literal_eval)

# Extract the first value from each list and store it as a float
df['mch_pre_float'] = df['mch_pre'].apply(lambda x: x[0] if isinstance(x, list) else x)
df['mch_post_float'] = df['mch_post'].apply(lambda x: x[0] if isinstance(x, list) else x)


In [ ]:
ind_df = pd.read_csv(ind_csv).drop(['Unnamed: 0'],axis=1)
# ind_df['mch_pre_float'] = ind_df['mch_pre'].apply(lambda x: x[0] if isinstance(x, list) else x)
# ind_df['mch_post_float'] = ind_df['mch_post'].apply(lambda x: x[0] if isinstance(x, list) else x)
ind_df

In [ ]:
for col in ind_df[['Baseline','Recall1','Recall2']]:
    ind_df[col] = ind_df[col].astype(int)

## Get mCherry intensity for ROIs 

In [ ]:
TSeries_all = [info['TSeries_g'].loc[(info['Session']==session)&(info['Animal']==ani)&(info['FOV']==fov)].values[0] for session in sessions]
cnms = [load_cnmf(os.path.join(base_dir,'GCaMP',ani,TSeries)) for TSeries in TSeries_all]


 # Initialize lists for storing the result
 # 
mch_pre = []
mch_post = []
tagged = []

# Iterate over each row in the ind_df DataFrame
for index, row in ind_df.iterrows():
    # Try to get the index from Baseline, Recall1, or Recall2 if available
    
    if row['Recall1'] != -1:
        idx = row['Recall1']  # Use Recall1 if Baseline is missing
        cnm = cnms[1]  # Corresponding CNMF model for Recall1
        session_post='Recall1'
    elif row['Recall2'] != -1:
        idx = row['Recall2']  # Use Recall2 if both Baseline and Recall1 are missing
        cnm = cnms[2]  # Corresponding CNMF model for Recall2
        session_post='Recall2'
    elif row['Baseline'] != -1:
        idx = row['Baseline']  # Use Baseline if present
        cnm = cnms[0]  # Corresponding CNMF model for Baseline
        session_post='Recall1'
        
    mch_im_pre = load_mch_image(ani,fov,'Baseline',file_key,base_dir,channel='red')
   
    # Calculate the intensity for both pre and post using the corresponding images
    if session_post=='Recall1':
        mch_im_post = load_mch_image(ani,fov,'Recall1',file_key,base_dir,channel='red')
        mch_int_post = mch_int(cnm, mch_im_post, idx=idx)  # Intensity on Recall1 (mch_im_post)
    if session_post=='Recall2':
        mch_im_post = load_mch_image(ani,fov,'Recall2',file_key,base_dir,channel='red')
        mch_int_post = mch_int(cnm, mch_im_post, idx=idx)  # Intensity on Recall1 (mch_im_post)
        
    mch_int_pre = mch_int(cnm, mch_im_pre, idx=idx)    # Intensity on Baseline (mch_im_pre)

    mch_pre.append(mch_int_pre)
    mch_post.append(mch_int_post)

    if mch_int_post>=thr:
        tagged.append(1)
    else:
        tagged.append(0)
        
    
# Convert the tagged results into DataFrame format
tagged_df = pd.DataFrame({'Tagged':tagged})
# Create DataFrames for mch_pre and mch_post
df_pre = pd.DataFrame(mch_pre, columns=['mch_pre'])
df_post = pd.DataFrame(mch_post, columns=['mch_post'])

In [ ]:
ind_df_new = pd.concat([ind_df,tagged_df,df_pre,df_post],axis=1)
ind_df_new

#### Write to fiji ROIs 

In [ ]:
import numpy as np
import roifile 

def write_roi_zip(coords_all,idxs,path,name,ani,fov,session):
    
    if idxs is  None:
        idxs = np.arange(0,len(coords_all))
    
    fname = os.path.join(path,ani+'_'+fov+'_'+session+'_'+name+'.zip')
    
    for idx in idxs: 
        coord = coords_all[idx]
        # good_cord = coord[coord]
        roi = roifile.ImagejRoi.frompoints(coord)
        roi.tofile(fname,name=str(idx))
    
    return fname

In [ ]:
# For each session
for s,session in enumerate(sessions):
    # Choose the corresponding CNMF model based on the session
    cnm = cnms[s]  # Select the CNMF model for that session

    # Get contours based on the current CNMF model
    coord_all = get_contours(cnm.estimates.A, dims=cnm.estimates.dims)
    coords = []
    for a in range(len(coord_all)):
        coord_a = coord_all[a]['coordinates'][1:-1]
        shape = coord_a[~np.isnan(coord_a)].shape[0]
        coord = coord_a[~np.isnan(coord_a)].reshape(int(shape / 2), 2)
        coords.append(coord)

    # For Tagged ROIs (1) in the current session (Post or Baseline, etc.)
    tagged_rois = ind_df_new[session].loc[ind_df_new.Tagged == 1].values
    tagged_rois = tagged_rois[tagged_rois != -1]
    write_roi_zip(coords_all=coords, idxs=tagged_rois, path=fiji_roi_savepath, name=f'{session}-mch', ani=ani, fov=fov, session=session)

    # For Non-tagged ROIs (0) in the current session (Post or Baseline, etc.)
    non_tagged_rois = ind_df_new[session].loc[ind_df_new.Tagged == 0].values
    non_tagged_rois = non_tagged_rois[non_tagged_rois != -1]
    write_roi_zip(coords_all=coords, idxs=non_tagged_rois, path=fiji_roi_savepath, name=f'{session}-non', ani=ani, fov=fov, session=session)

In [ ]:
tagged_df

####  Manually fix the wrong falsely tagged/non-tagged

In [ ]:
false_pos_post = input('input False mcherry positives from Post rois').split(',')
false_pos_post

In [ ]:
false_neg_post = input('input False mcherry negatives from Post rois').split(',')
false_neg_post

In [ ]:
false_pos_pre = input('input False mcherry positives from Pre rois').split(',')
false_pos_pre

In [ ]:
false_neg_pre = input('input False mcherry negatives from Baseline rois').split(',')
false_neg_pre

optional - only if recall2

In [ ]:
false_neg_re2 = input('input False mcherry negatives from Recall2 rois').split(',')
false_neg_re2

In [ ]:
false_pos_re2 = input('input False mcherry positives from Recall2 rois').split(',')
false_pos_re2

## save any labeled false positives/negatives

In [ ]:
fn_pre = [int(x) for x in false_neg_pre]
fn_post = [int(x) for x in false_neg_post]
fp_pre = [int(x) for x in false_pos_pre]
fp_post = [int(x) for x in false_pos_post]

np.save(os.path.join(base_dir,'Tagging','Manual_correction_false_mcherry',ani+'_'+fov+'_manual_changes_FP_post.npy'),fp_post)
np.save(os.path.join(base_dir,'Tagging','Manual_correction_false_mcherry',ani+'_'+fov+'_manual_changes_FN_post.npy'),fn_post)
np.save(os.path.join(base_dir,'Tagging','Manual_correction_false_mcherry',ani+'_'+fov+'_manual_changes_FP_pre.npy'),fp_pre)
np.save(os.path.join(base_dir,'Tagging','Manual_correction_false_mcherry',ani+'_'+fov+'_manual_changes_FN_pre.npy'),fn_pre)


In [ ]:
fp_re2 = [int(x) for x in false_pos_re2]
fn_re2 = [int(x) for x in false_neg_re2]
np.save(os.path.join(base_dir,'Tagging','Manual_correction_false_mcherry',ani+'_'+fov+'_manual_changes_FP_re2.npy'),fp_re2)
np.save(os.path.join(base_dir,'Tagging','Manual_correction_false_mcherry',ani+'_'+fov+'_manual_changes_FN_re2.npy'),fn_re2)

#### load them in and resave the Cell Index DataFrame

In [ ]:
load=False
if load: 
    fn_pre = np.load(os.path.join(base_dir,'Manual_correction_false_mcherry','Tagging',ani+'_'+fov+'_manual_changes_FN_pre.npy'))
    fn_post = np.load(os.path.join(base_dir,'Manual_correction_false_mcherry','Tagging',ani+'_'+fov+'_manual_changes_FN_post.npy'))
    fp_pre =np.load(os.path.join(base_dir,'Manual_correction_false_mcherry','Tagging',ani+'_'+fov+'_manual_changes_FP_pre.npy'))
    fp_post = np.load(os.path.join(base_dir,'Manual_correction_false_mcherry','Tagging',ani+'_'+fov+'_manual_changes_FP_post.npy'))
    


In [ ]:
ind_df_new

## save final csv

In [ ]:
manual_df = ind_df_new.copy()
for fp in fp_post:
    manual_df.loc[manual_df.Recall1==fp,'Tagged']=0
for fn in fn_post:
    manual_df.loc[manual_df.Recall1==fn,'Tagged']=1
for fp in fp_pre:
    manual_df.loc[manual_df.Baseline==fp,'Tagged']=0
for fn in fn_pre:
    manual_df.loc[manual_df.Baseline==fn,'Tagged']=1

In [ ]:
manual_df = ind_df_new.copy()
for fp in fp_re2:
    manual_df.loc[manual_df.Recall2==fp,'Tagged']=0
for fn in fn_re2:
    manual_df.loc[manual_df.Recall2==fn,'Tagged']=1

In [ ]:
manual_df

In [ ]:
manual_df.to_csv(os.path.join(base_dir,'Tagging',ani+'_'+fov+'_indices_split.csv'))
manual_df